# Figure 5 — SVD alignment of the conv transfer (both donors)

For each conv layer we SVD the pretrained weight into U, Sigma, V and reset one factor to
the upstream init (aligned via Procrustes to the pristine donor). "conv only" is the ceiling
(FC/BN reset); "keep S&V" resets U to init; "keep U" resets Sigma,V to init. Two panels: the
random-label donor (linear-transfer regime, slope-1 reference) and the random-pixel donor
(no -1 regime, the reference is drawn for scale only). All t0 from `common.fit_t0`.

- **Consumes** (ingested metrics):
  `surgery_svd_t0_labelseed42/<donor>/knockout_GF`,
  `surgery_svd_fcbn_reset_labelseed42/<donor>/{reset_u_uinit_aligned, reset_sv_svinit_aligned}`
  for donor in {randlabel, randpixel}
- **Produces:** `outputs/fig5_svd_alignment.png`


In [ ]:
import sys
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# repo root (dir holding common/) onto sys.path, from any working directory
_root = Path.cwd()
while not (_root / "common").is_dir() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from common.config import FIGURE_DATA_DIR, OUTPUT_DIR
from common.fit_t0 import fit_t0

KGRID = {"randlabel": [0, 5, 10, 15, 20, 25, 30, 35, 40, 45], "randpixel": [0, 5, 10, 15]}

SV = FIGURE_DATA_DIR / "surgery_svd_t0_labelseed42"            # knockout_GF = conv-only ceiling
FB = FIGURE_DATA_DIR / "surgery_svd_fcbn_reset_labelseed42"    # reset_u / reset_sv (aligned)


def t0_of(metrics_path):
    m = json.loads(Path(metrics_path).read_text(encoding="utf-8"))
    return fit_t0([r["epoch"] for r in m["epochs"]], [r["train_acc"] for r in m["epochs"]]).t0


def curve(cond_dir, kg):
    return np.array([t0_of(cond_dir / f"epoch{k:04d}" / "metrics.json") for k in kg], float)

In [ ]:
C_CONV, C_SV, C_U = "#1a1a1a", "#6baed6", "#fdae6b"     # black / light blue / light orange


def lines_for(up):
    return [("conv only", SV / up / "knockout_GF", C_CONV, "-o"),
            ("keep S&V", FB / up / "reset_u_uinit_aligned", C_SV, "-s"),
            ("keep U",   FB / up / "reset_sv_svinit_aligned", C_U, "-^")]


ups = ("randlabel", "randpixel")
data, allv = {}, []
for up in ups:
    kg = KGRID[up]
    rows = [(lab, curve(d, kg), col, st) for lab, d, col, st in lines_for(up)]
    data[up] = (kg, rows)
    allv += [v for _, arr, _, _ in rows for v in arr]
Y_LO = math.floor(min(allv) / 5) * 5
Y_HI = math.ceil(max(allv) / 5) * 5
Y_SPAN = Y_HI - Y_LO
X_LIMS = {up: (-2.0, max(KGRID[up]) + 2.0) for up in ups}
X_SPANS = {up: X_LIMS[up][1] - X_LIMS[up][0] for up in ups}
SCALE = 0.145
LEFT, BOTTOM, GAP = 0.75, 0.6, 1.0
H = Y_SPAN * SCALE
widths = {"randlabel": X_SPANS["randlabel"] * SCALE}
widths["randpixel"] = widths["randlabel"] * 0.60
FIG_W = LEFT + widths["randlabel"] + GAP + widths["randpixel"] + 2.2
FIG_H = BOTTOM + H + 0.45
fig = plt.figure(figsize=(FIG_W, FIG_H))
axes, x0 = {}, LEFT
for up in ups:
    axes[up] = fig.add_axes([x0 / FIG_W, BOTTOM / FIG_H, widths[up] / FIG_W, H / FIG_H])
    x0 += widths[up] + GAP
for up in ups:
    ax = axes[up]
    kg, rows = data[up]
    # from-scratch baseline (epoch-0 anchor) + slope -1 reference, in BOTH panels
    y0 = max(arr[0] for _, arr, _, _ in rows if not np.isnan(arr[0]))
    ax.plot([0.0, X_LIMS[up][1]], [y0, y0], color="#c44e52", ls="--", lw=1.2, zorder=1,
            label="From-scratch baseline" if up == "randlabel" else None)
    x_end = min(y0 - Y_LO, X_LIMS[up][1])
    ax.plot([0.0, x_end], [y0, y0 - x_end], color="0.45", ls="--", lw=1.3, zorder=1,
            label="slope = \u22121 (ref.)" if up == "randlabel" else None)
    for lab, arr, col, st in rows:
        ax.plot(kg, arr, st, color=col, alpha=0.9, label=lab, mec="white", mew=0.5, ms=6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Pre-training epoch", fontsize=12)
    ax.set_title(f"({'a' if up == 'randlabel' else 'b'}) {up} \u2192 random label", fontsize=12, pad=8)
    ax.tick_params(labelsize=10)
    ax.set_xticks(kg)
    ax.set_xlim(*X_LIMS[up])
    ax.set_ylim(Y_LO, Y_HI)
axes["randlabel"].set_ylabel("$t_0$ (downstream epochs)", fontsize=12)
axes["randlabel"].legend(fontsize=8.5, frameon=False, loc="lower left", handlelength=1.9, borderaxespad=0.4)

OUT_PNG = OUTPUT_DIR / "fig5_svd_alignment.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=400, bbox_inches="tight")
print("saved", OUT_PNG)